<a href="https://colab.research.google.com/github/Hoshino-lab/Ichijo_2026_scRNAseq/blob/main/Seizure_autodetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mouse EEG seizure detection in Google Colab — Python version

This notebook mounts Google Drive, loads:

1. an EEG CSV from Drive, and
2. a **user-selected baseline-region CSV** from Drive,

then runs a **median/MAD Hilbert-envelope seizure detector**.

The important change in this version: **baseline is selected by the user**, not automatically estimated from the lower 80% of the recording.

## EEG CSV format

Your EEG CSV should have one row per EEG sample.

Minimum required columns:

- `time_s`: time in seconds
- `eeg_uV`: EEG amplitude, preferably in microvolts

Optional validation columns:

- `truth_seizure`: `1` during known/simulated seizure, otherwise `0`
- `truth_spike`: `1` during known/simulated interictal spike, otherwise `0`

Example EEG CSV:

```csv
time_s,eeg_uV,truth_seizure,truth_spike
0.00,12.4,0,0
0.01,15.8,0,0
0.02,9.7,0,0
0.03,-3.1,0,0
1800.00,502.6,0,1
5400.00,210.3,1,0
5400.01,286.9,1,0
5400.02,318.2,1,0
5430.00,18.5,0,0
```

## User-selected baseline CSV format

Create a separate CSV containing seizure-free baseline intervals chosen by the user.

Required columns:

- `start_s`: baseline start time in seconds
- `end_s`: baseline end time in seconds

Optional columns such as `description` are allowed and ignored by the detector.

Example baseline CSV:

```csv
start_s,end_s,description
60,300,quiet seizure-free baseline
1200,1500,seizure-free baseline
3600,3900,seizure-free baseline
```

The detector computes the Hilbert envelope for the full EEG, but calculates the baseline median and MAD **only from samples inside these selected baseline windows**.

For a 100 Hz toy dataset, the high bandpass cutoff is automatically capped below Nyquist, so the effective bandpass becomes about **1–45 Hz**. For real mouse EEG sampled at 1000 Hz, it uses **1–100 Hz**.


## 1. Mount Google Drive

Upload both CSV files to Google Drive first:

- EEG data CSV
- baseline regions CSV


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Set file paths

Change these paths to match your files in Google Drive.

In [ ]:
from pathlib import Path

# EEG CSV on Google Drive.
eeg_csv_path = '/content/drive/MyDrive/mouse_EEG_2h_simulation_with_MAD_detection.csv'

# User-selected baseline-region CSV on Google Drive.
baseline_csv_path = '/content/drive/MyDrive/baseline_regions.csv'

# Results will be saved here.
output_dir = '/content/drive/MyDrive/MAD_EEG_detection_outputs_user_baseline'
Path(output_dir).mkdir(parents=True, exist_ok=True)

print('EEG CSV:', eeg_csv_path)
print('Baseline CSV:', baseline_csv_path)
print('Output folder:', output_dir)


## Optional: upload CSV files directly to Colab instead of Drive

Use this only if you do not want to store the files in Drive. After uploading, set `eeg_csv_path` and `baseline_csv_path` to the uploaded filenames.

In [ ]:
# Optional file picker. Uncomment to use.
# from google.colab import files
# uploaded = files.upload()
# print(uploaded.keys())
# eeg_csv_path = 'your_uploaded_eeg_file.csv'
# baseline_csv_path = 'your_uploaded_baseline_regions.csv'


## 3. Install/import packages


In [ ]:
import importlib.util
import sys
import subprocess

for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert

print('Packages loaded.')


## 4. Define detector functions


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert


def infer_fs(time_s):
    """Infer sampling frequency from time_s."""
    time_s = np.asarray(time_s, dtype=float)
    dt = np.median(np.diff(time_s))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Could not infer sampling rate from time_s.")
    return 1.0 / dt


def boolean_runs(mask):
    """Return start/end sample indices for True runs."""
    mask = np.asarray(mask, dtype=bool)
    changes = np.diff(mask.astype(int), prepend=0, append=0)
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1)
    return starts, ends


def bandpass_filter(eeg, fs, low_hz=1.0, high_hz=100.0, order=4):
    """
    Bandpass filter EEG.
    high_hz is automatically capped below Nyquist.
    """
    nyquist = fs / 2.0
    effective_high = min(high_hz, nyquist * 0.90)

    if effective_high <= low_hz:
        raise ValueError(
            f"Invalid bandpass: low_hz={low_hz}, effective_high_hz={effective_high}, fs={fs}"
        )

    sos = butter(
        order,
        [low_hz, effective_high],
        btype="bandpass",
        fs=fs,
        output="sos",
    )
    filtered = sosfiltfilt(sos, eeg)
    return filtered, effective_high


def moving_average(x, window_s, fs):
    """Centered moving average."""
    n = max(1, int(round(window_s * fs)))
    kernel = np.ones(n) / n
    return np.convolve(x, kernel, mode="same")


def baseline_mask_from_regions(time_s, baseline_regions_df, start_col="start_s", end_col="end_s"):
    """
    Build a boolean baseline mask from user-selected baseline intervals.

    baseline_regions_df must contain:
      start_s,end_s

    Each row marks a seizure-free baseline epoch.
    """
    if start_col not in baseline_regions_df.columns:
        raise ValueError(f"Baseline CSV is missing required column: {start_col}")
    if end_col not in baseline_regions_df.columns:
        raise ValueError(f"Baseline CSV is missing required column: {end_col}")

    time_s = np.asarray(time_s, dtype=float)
    mask = np.zeros(time_s.shape, dtype=bool)

    for i, row in baseline_regions_df.iterrows():
        start_s = float(row[start_col])
        end_s = float(row[end_col])

        if not np.isfinite(start_s) or not np.isfinite(end_s):
            raise ValueError(f"Baseline row {i} contains non-finite start/end values.")
        if end_s <= start_s:
            raise ValueError(f"Baseline row {i} has end_s <= start_s.")

        mask |= (time_s >= start_s) & (time_s <= end_s)

    if not np.any(mask):
        raise ValueError("No EEG samples fell inside the selected baseline regions.")

    return mask


def robust_baseline_from_user_regions(smooth_envelope, baseline_mask):
    """
    Compute baseline median and MAD only from user-selected baseline samples.
    """
    baseline_values = np.asarray(smooth_envelope)[np.asarray(baseline_mask, dtype=bool)]

    if baseline_values.size < 10:
        raise ValueError(
            f"Too few baseline samples: {baseline_values.size}. Select longer baseline regions."
        )

    median = np.median(baseline_values)
    mad = np.median(np.abs(baseline_values - median))

    # Convert MAD to Gaussian-equivalent robust sigma.
    robust_sigma = max(1.4826 * mad, 1e-12)

    return median, robust_sigma, baseline_values.size


def merge_boolean_events(mask, fs, merge_gap_s=2.0):
    """Merge True runs separated by <= merge_gap_s."""
    mask = np.asarray(mask, dtype=bool)
    starts, ends = boolean_runs(mask)

    if len(starts) <= 1:
        return mask.copy()

    out = np.zeros_like(mask, dtype=bool)
    max_gap_samples = int(round(merge_gap_s * fs))

    current_start = starts[0]
    current_end = ends[0]

    for s, e in zip(starts[1:], ends[1:]):
        if s - current_end <= max_gap_samples:
            current_end = e
        else:
            out[current_start:current_end] = True
            current_start = s
            current_end = e

    out[current_start:current_end] = True
    return out


def apply_min_duration(mask, fs, min_duration_s=5.0):
    """Keep only True runs lasting at least min_duration_s."""
    starts, ends = boolean_runs(mask)
    out = np.zeros_like(mask, dtype=bool)
    min_samples = int(round(min_duration_s * fs))

    for s, e in zip(starts, ends):
        if e - s >= min_samples:
            out[s:e] = True

    return out


def detect_seizures_mad_user_baseline(
    eeg,
    time_s,
    baseline_regions_df,
    bandpass_low_hz=1.0,
    bandpass_high_hz=100.0,
    smooth_window_s=1.0,
    threshold_z=5.0,
    min_duration_s=5.0,
    merge_gap_s=2.0,
):
    """
    Median/MAD Hilbert-envelope seizure detector using user-selected baseline regions.
    """
    eeg = np.asarray(eeg, dtype=float)
    time_s = np.asarray(time_s, dtype=float)
    fs = infer_fs(time_s)

    filtered, effective_high_hz = bandpass_filter(
        eeg,
        fs,
        low_hz=bandpass_low_hz,
        high_hz=bandpass_high_hz,
    )

    envelope = np.abs(hilbert(filtered))
    smooth_envelope = moving_average(envelope, smooth_window_s, fs)

    baseline_mask = baseline_mask_from_regions(time_s, baseline_regions_df)
    baseline_median, baseline_robust_sigma, baseline_sample_count = robust_baseline_from_user_regions(
        smooth_envelope,
        baseline_mask,
    )

    z = (smooth_envelope - baseline_median) / baseline_robust_sigma

    above_threshold = z > threshold_z
    merged = merge_boolean_events(above_threshold, fs, merge_gap_s=merge_gap_s)
    detected = apply_min_duration(merged, fs, min_duration_s=min_duration_s)

    starts, ends = boolean_runs(detected)

    events = []
    for event_id, (s, e) in enumerate(zip(starts, ends), start=1):
        events.append({
            "event_id": event_id,
            "onset_s": s / fs,
            "offset_s": e / fs,
            "duration_s": (e - s) / fs,
            "onset_min": (s / fs) / 60.0,
            "offset_min": (e / fs) / 60.0,
            "duration_min": ((e - s) / fs) / 60.0,
            "max_z": float(np.max(z[s:e])),
            "mean_z": float(np.mean(z[s:e])),
        })

    events_df = pd.DataFrame(events)

    params = {
        "fs_hz": fs,
        "bandpass_low_hz": bandpass_low_hz,
        "requested_bandpass_high_hz": bandpass_high_hz,
        "effective_bandpass_high_hz": effective_high_hz,
        "smooth_window_s": smooth_window_s,
        "threshold_z": threshold_z,
        "min_duration_s": min_duration_s,
        "merge_gap_s": merge_gap_s,
        "baseline_sample_count": int(baseline_sample_count),
        "baseline_duration_s": float(baseline_sample_count / fs),
        "baseline_median": float(baseline_median),
        "baseline_robust_sigma": float(baseline_robust_sigma),
    }

    return {
        "fs": fs,
        "filtered": filtered,
        "envelope": envelope,
        "smooth_envelope": smooth_envelope,
        "baseline_mask": baseline_mask,
        "z": z,
        "detected": detected,
        "events": events_df,
        "params": params,
    }


def validate_against_truth(detected, truth, fs):
    """Time-sample validation against a binary truth vector."""
    detected = np.asarray(detected, dtype=bool)
    truth = np.asarray(truth, dtype=bool)

    tp = np.sum(detected & truth)
    fp = np.sum(detected & ~truth)
    fn = np.sum(~detected & truth)

    sensitivity = tp / max(tp + fn, 1)
    precision = tp / max(tp + fp, 1)

    return {
        "true_burden_s": float(np.sum(truth) / fs),
        "detected_burden_s": float(np.sum(detected) / fs),
        "time_sample_sensitivity": float(sensitivity),
        "time_sample_precision": float(precision),
    }


def add_region_spans(ax, regions_df, start_col="start_s", end_col="end_s", alpha=0.12):
    """Shade regions on a plot."""
    for _, row in regions_df.iterrows():
        ax.axvspan(float(row[start_col]) / 60.0, float(row[end_col]) / 60.0, alpha=alpha)


def plot_eeg_overview(time_s, eeg, events_df, baseline_regions_df, out_path, decimate=10):
    """Save whole-recording EEG overview plot."""
    time_s = np.asarray(time_s)
    eeg = np.asarray(eeg)
    decimate = max(1, int(decimate))

    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(time_s[::decimate] / 60.0, eeg[::decimate], linewidth=0.8)
    add_region_spans(ax, baseline_regions_df, alpha=0.10)

    if events_df is not None and len(events_df) > 0:
        for _, row in events_df.iterrows():
            ax.axvspan(row["onset_min"], row["offset_min"], alpha=0.25)

    ax.set_xlabel("Time (min)")
    ax.set_ylabel("EEG")
    ax.set_title("EEG overview: baseline regions and detected seizure events")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_z_overview(time_s, z, events_df, baseline_regions_df, threshold_z, out_path, decimate=10):
    """Save robust z-score overview plot."""
    time_s = np.asarray(time_s)
    z = np.asarray(z)
    decimate = max(1, int(decimate))

    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(time_s[::decimate] / 60.0, z[::decimate], linewidth=0.8)
    ax.axhline(threshold_z, linestyle="--", linewidth=1)
    add_region_spans(ax, baseline_regions_df, alpha=0.10)

    if events_df is not None and len(events_df) > 0:
        for _, row in events_df.iterrows():
            ax.axvspan(row["onset_min"], row["offset_min"], alpha=0.25)

    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Robust envelope z")
    ax.set_title("Median/MAD detector using user-selected baseline")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)


def plot_event_zooms(time_s, eeg, events_df, out_dir, pad_s=10.0):
    """Save one zoomed EEG plot per detected event."""
    out_paths = []
    time_s = np.asarray(time_s)
    eeg = np.asarray(eeg)

    if events_df is None or len(events_df) == 0:
        return out_paths

    for _, row in events_df.iterrows():
        event_id = int(row["event_id"])
        onset_s = float(row["onset_s"])
        offset_s = float(row["offset_s"])

        mask = (time_s >= onset_s - pad_s) & (time_s <= offset_s + pad_s)

        fig, ax = plt.subplots(figsize=(12, 4.5))
        ax.plot(time_s[mask] / 60.0, eeg[mask], linewidth=0.9)
        ax.axvspan(row["onset_min"], row["offset_min"], alpha=0.25)
        ax.set_xlabel("Time (min)")
        ax.set_ylabel("EEG")
        ax.set_title(f"Detected event {event_id}: EEG zoom")
        fig.tight_layout()

        out_path = Path(out_dir) / f"detected_event_{event_id:03d}_zoom.png"
        fig.savefig(out_path, dpi=180)
        plt.close(fig)

        out_paths.append(out_path)

    return out_paths


def run_detector_from_csv_user_baseline(
    eeg_csv_path,
    baseline_csv_path,
    output_dir,
    eeg_column="eeg_uV",
    time_column="time_s",
    threshold_z=5.0,
    min_duration_s=5.0,
    merge_gap_s=2.0,
    smooth_window_s=1.0,
    bandpass_low_hz=1.0,
    bandpass_high_hz=100.0,
):
    """
    Full pipeline with user-selected baseline regions:
    1. Load EEG CSV
    2. Load baseline-region CSV
    3. Compute median/MAD from those baseline regions only
    4. Detect events
    5. Save processed CSV, events CSV, summary TXT, and plots
    """
    eeg_csv_path = Path(eeg_csv_path)
    baseline_csv_path = Path(baseline_csv_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    eeg_df = pd.read_csv(eeg_csv_path)
    baseline_regions_df = pd.read_csv(baseline_csv_path)

    if time_column not in eeg_df.columns:
        raise ValueError(f"EEG CSV missing required column: {time_column}")
    if eeg_column not in eeg_df.columns:
        raise ValueError(f"EEG CSV missing required column: {eeg_column}")

    time_s = eeg_df[time_column].to_numpy(dtype=float)
    eeg = eeg_df[eeg_column].to_numpy(dtype=float)

    result = detect_seizures_mad_user_baseline(
        eeg=eeg,
        time_s=time_s,
        baseline_regions_df=baseline_regions_df,
        bandpass_low_hz=bandpass_low_hz,
        bandpass_high_hz=bandpass_high_hz,
        smooth_window_s=smooth_window_s,
        threshold_z=threshold_z,
        min_duration_s=min_duration_s,
        merge_gap_s=merge_gap_s,
    )

    fs = result["fs"]
    events_df = result["events"]

    eeg_df["filtered_eeg"] = result["filtered"]
    eeg_df["hilbert_envelope"] = result["envelope"]
    eeg_df["smoothed_envelope"] = result["smooth_envelope"]
    eeg_df["baseline_mask"] = result["baseline_mask"].astype(int)
    eeg_df["robust_z"] = result["z"]
    eeg_df["detected_seizure"] = result["detected"].astype(int)

    processed_csv = output_dir / "processed_EEG_with_user_baseline_MAD_detection.csv"
    events_csv = output_dir / "MAD_detected_events_user_baseline.csv"
    summary_txt = output_dir / "MAD_detection_summary_user_baseline.txt"
    baseline_used_csv = output_dir / "baseline_regions_used.csv"
    eeg_overview_png = output_dir / "EEG_overview_user_baseline_with_detections.png"
    z_overview_png = output_dir / "MAD_zscore_user_baseline_overview.png"

    eeg_df.to_csv(processed_csv, index=False)
    events_df.to_csv(events_csv, index=False)
    baseline_regions_df.to_csv(baseline_used_csv, index=False)

    validation = None
    if "truth_seizure" in eeg_df.columns:
        validation = validate_against_truth(
            detected=result["detected"],
            truth=eeg_df["truth_seizure"].to_numpy(dtype=bool),
            fs=fs,
        )

    decimate = max(1, int(round(fs / 10)))  # about 10 samples/s for overview plotting
    plot_eeg_overview(time_s, eeg, events_df, baseline_regions_df, eeg_overview_png, decimate=decimate)
    plot_z_overview(time_s, result["z"], events_df, baseline_regions_df, threshold_z, z_overview_png, decimate=decimate)
    zoom_paths = plot_event_zooms(time_s, eeg, events_df, output_dir, pad_s=10.0)

    lines = [
        "Median/MAD Hilbert-envelope seizure detection summary",
        "Using user-selected baseline regions",
        "====================================================",
        f"EEG CSV: {eeg_csv_path}",
        f"Baseline CSV: {baseline_csv_path}",
        f"Output folder: {output_dir}",
        "",
        "Baseline CSV format:",
        "  start_s,end_s",
        "",
        "Parameters:",
    ]

    for key, value in result["params"].items():
        if isinstance(value, float):
            lines.append(f"  {key}: {value:.6g}")
        else:
            lines.append(f"  {key}: {value}")

    lines.extend([
        "",
        f"Detected events: {len(events_df)}",
    ])

    if len(events_df) > 0:
        for _, row in events_df.iterrows():
            lines.append(
                f"  Event {int(row['event_id'])}: "
                f"onset {row['onset_min']:.3f} min, "
                f"offset {row['offset_min']:.3f} min, "
                f"duration {row['duration_s']:.2f} s, "
                f"max z {row['max_z']:.2f}"
            )

    if validation is not None:
        lines.extend(["", "Validation against truth_seizure column:"])
        for key, value in validation.items():
            lines.append(f"  {key}: {value:.6g}")

    lines.extend([
        "",
        "Output files:",
        f"  {processed_csv}",
        f"  {events_csv}",
        f"  {baseline_used_csv}",
        f"  {summary_txt}",
        f"  {eeg_overview_png}",
        f"  {z_overview_png}",
    ])

    for p in zoom_paths:
        lines.append(f"  {p}")

    summary = "\n".join(lines)
    summary_txt.write_text(summary)

    return {
        "eeg_data": eeg_df,
        "baseline_regions": baseline_regions_df,
        "events": events_df,
        "result": result,
        "validation": validation,
        "paths": {
            "processed_csv": processed_csv,
            "events_csv": events_csv,
            "baseline_used_csv": baseline_used_csv,
            "summary_txt": summary_txt,
            "eeg_overview_png": eeg_overview_png,
            "z_overview_png": z_overview_png,
            "zoom_pngs": zoom_paths,
        },
        "summary": summary,
    }


## 5. Preview input files

This checks that the EEG CSV and baseline CSV have the expected columns.

In [ ]:
eeg_preview = pd.read_csv(eeg_csv_path, nrows=5)
baseline_preview = pd.read_csv(baseline_csv_path)

print('EEG CSV preview:')
display(eeg_preview)

print('Baseline regions CSV:')
display(baseline_preview)


## 6. Run detection using user-selected baseline regions

Default settings:

- bandpass: 1–100 Hz, automatically capped below Nyquist
- Hilbert envelope
- 1 s smoothing
- median/MAD calculated only from the uploaded baseline regions
- robust z threshold: `z > 5`
- minimum event duration: 5 s
- merge events separated by ≤2 s


In [ ]:
analysis = run_detector_from_csv_user_baseline(
    eeg_csv_path=eeg_csv_path,
    baseline_csv_path=baseline_csv_path,
    output_dir=output_dir,
    eeg_column='eeg_uV',
    time_column='time_s',
    threshold_z=5.0,
    min_duration_s=5.0,
    merge_gap_s=2.0,
    smooth_window_s=1.0,
    bandpass_low_hz=1.0,
    bandpass_high_hz=100.0,
)

print(analysis['summary'])


## 7. View detected events


In [ ]:
analysis['events']


## 8. Display output plots


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(analysis['paths']['eeg_overview_png'])))
display(Image(filename=str(analysis['paths']['z_overview_png'])))

for p in analysis['paths']['zoom_pngs']:
    display(Image(filename=str(p)))


## 9. Tuning notes

If the detector misses seizures, try lowering `threshold_z` to 4.0 or 3.5, shortening `smooth_window_s`, or lowering `min_duration_s`.

If it overcalls artifacts, try increasing `threshold_z`, increasing `min_duration_s`, or selecting cleaner baseline regions.

The baseline CSV matters a lot: avoid periods containing seizures, large movement artifacts, electrode noise, or postictal suppression unless those are intentionally part of your baseline definition.

In [ ]:
# Example: rerun with a less conservative threshold
# analysis_threshold4 = run_detector_from_csv_user_baseline(
#     eeg_csv_path=eeg_csv_path,
#     baseline_csv_path=baseline_csv_path,
#     output_dir='/content/drive/MyDrive/MAD_EEG_detection_outputs_user_baseline_threshold4',
#     threshold_z=4.0,
#     min_duration_s=5.0,
#     merge_gap_s=2.0,
#     smooth_window_s=1.0,
#     bandpass_low_hz=1.0,
#     bandpass_high_hz=100.0,
# )
# print(analysis_threshold4['summary'])
